# Ubicar nuevos pozos

# Contenido <a id='back'></a>

* [Introducción](#intro)
* [Etapa 1. Descripción de los datos](#data_review)
* [Etapa 2. Modelo de Machine Learning](#Models)
    * [2.1 Modelo de regresion lineal para predicción de volumen de reserva de pozos](#linear_regression)
    * [2.2 Cálculo de ganancias](#gains)
* [Conclusiones](#end)

# Introducción <a id='intro'></a>      

La compañia OilyGiant se dedida a la extracción de petróleo y actualmente se plantea la prediccion de la región que genera mayor ganancia para crear 200 nuevos pozos, se analizan las caracteristicas calidad de crudo y volumen de reservas, que han determinado pozos exitosos en el pasado, de manera que con estos datos se entrena un modelo de regresion lineal que predice el volumen de reservas en pozos nuevos, se eligen los pozos petrolíferos que tienen los valores estimados más altos para de esta forma elegir la región con el beneficio total más alto para los pozos petrolíferos seleccionados. Otra cuestión que se plantea es el costo de estos nuevos pozos frente a la ganancia por lo cual se hace el cálculo financiero acerca de la inversion incial y cuanto es la ganancia por miles de barriles, para conocer el número de barriles esperados para tener una ganacia o perdida. De esta forma se contempla cual region tiene mayor ganancia con repecto al costo para asi tener el mayor margen de beficio. 


Condiciones que se tuvieron en cuenta 
Al explorar la región, se lleva a cabo un estudio de 500 puntos con la selección de los mejores 200 puntos para el cálculo del beneficio.
El presupuesto para el desarrollo de 200 pozos petroleros es de 100 millones de dólares.
Un barril de materias primas genera 4.5 USD de ingresos. El ingreso de una unidad de producto es de 4500 dólares (el volumen de reservas está expresado en miles de barriles).
Después de la evaluación de riesgo, se mantén solo las regiones con riesgo de pérdidas inferior al 2.5%. De las que se ajustan a los criterios, se  seleccionó la región con el beneficio promedio más alto.
Los datos son sintéticos: los detalles del contrato y las características del pozo no se publican.

# Etapa 1. Descripción de los datos <a id='data_review'></a>

In [10]:
#importamos librerias que vamos a usar
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [11]:
#lectura de los datasets que usaremos, de las 3 regiones de pozos
df_0=pd.read_csv('geo_data_0.csv') 
df_1=pd.read_csv('geo_data_1.csv')
df_2=pd.read_csv('geo_data_2.csv')

In [12]:
#Funcion para observar los datos, info, mediaa, maxima, duplicados, nulos
def observar_datos(data, head=5):
    print("##################### info #####################")
    print()
    print(data.info())
    print("##################### sample #####################")
    print()
    print(data.sample(head))
    print("##################### describe #####################")
    print()
    print(data.describe())
    print("##################### duplicados #####################")
    print()
    print(data.duplicated().sum())
    print("##################### nulos #####################")
    print()
    print(data.isna().sum())

In [13]:
#aplicacion de funcion para prepar datos en el primer dataset
observar_datos(df_0)

##################### info #####################

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
##################### sample #####################

          id        f0        f1        f2     product
62425  5MAJl -0.820277  0.438595 -0.372787  121.013067
57810  XLoJY  1.509065 -0.546656  3.616841   82.603422
57074  qP6c3  0.084869  1.095583 -0.259258   36.192866
29909  J22Kr  0.037285  0.224338 -0.924724   51.443128
14793  m2XVy  0.599620  0.519449  5.698790  112.266992
##################### describe #####################

                  f0             f1             f2        product
count  10

Observamos que no se tienen datos faltantes ni duplicados, este data set esta listo para el modelo de predicción. 
 Las columnas se definen: 
 
 id — identificador único de pozo de petróleo
 
 f0, f1, f2 — tres características de los puntos (su significado específico no es importante, pero las características en sí son significativas)
 
 product — volumen de reservas en el pozo de petróleo (miles de barriles).

In [14]:
#aplicacion de funcion para prepar datos en el segundo dataset
observar_datos(df_1)

##################### info #####################

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
##################### sample #####################

          id         f0         f1        f2     product
91593  H0oHq   4.410975   1.596878  2.006055   53.906522
66216  rxB4j  14.004577 -12.169723  3.995208  107.813044
76193  KUspj  12.972530  -8.629411  0.994091   26.953261
34701  8IDOD -11.843303 -10.331792  2.995100   84.038886
4309   39oI9  -4.551069  -8.166754  4.995670  137.945408
##################### describe #####################

                  f0             f1             f2        produ

In [15]:
#aplicacion de funcion para prepar datos en el tercer dataset
observar_datos(df_2)

##################### info #####################

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
##################### sample #####################

          id        f0        f1        f2     product
88207  R07tT  0.350489  0.693597  5.540417  125.090912
18253  qpALu  0.830682  2.530280  5.899553  124.948662
6475   AnIou -0.713105  0.671283 -0.944785  101.818851
41353  ZFFg7  1.479825  0.829569  7.442547  117.116821
62477  grz5p -4.121272  3.565874  7.969917  101.363121
##################### describe #####################

                  f0             f1             f2        product
count  10

Observamos que cada region contiene 100,000 pozos, los valores de f0,f1,f2 ya estan normalizados, y no se cuenta con valores nulos en los 3 datasets. 

id — identificador único de pozo de petróleo

product — volumen de reservas en el pozo de petróleo (miles de barriles).

f0, f1, f2 — tres características de los puntos

# Etapa 2. Modelo de Machine Learning <a id='Models'></a>

## Modelo de regresion lineal para predicción de volumen de reserva de pozos <a id='linear_regression'></a>

In [22]:
#Debido a que es una prediccion cuantitativa se toma el modelo de regresion lineal
state=np.random.RandomState(12345)
def crear_modelo_entrenar_predecir(data, state):
  target = data['product']
  features = data.drop(['product', 'id'], axis=1)

  # segmentación en datos de entrenamiento, validacion y prueba
  features_train,features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.25, random_state=state)
  model = LinearRegression() # inicializa el constructor de modelos
  model.fit(features_train, target_train) # entrena el modelo en el conjunto de entrenamiento
  predictions = model.predict(features_valid) # obtén las predicciones del modelo en el conjunto de validación

   # calcula la RECM en el conjunto de validación.
   #la función mean_squared_error calcula el error medio al cuadrado, pero para la unidad de medida de los barriles de petroleo es incoherente
   #expresarla al cuadrado, entonces calculamos la raíz cuadrada.
  rmse =np.sqrt(mean_squared_error(target_valid, predictions))

  #Muestra en la pantalla el volumen promedio de las reservas previstas y el RECM del modelo
  print("RMSE", rmse)

  print("Volumen de barriles promedio predecidos", predictions.mean())
  print("Volumen de barriles promedio reales", target_valid.mean())
    
    
  #Guarda las predicciones y las respuestas correctas para el conjunto de validación
  resultado=pd.DataFrame(data={'valor_real':target_valid, 'prediccion_valores':predictions,})
  return resultado

In [23]:
#Creamos el modelo de regresión lineal para cada REGION
regiones= ['Region 1', 'Region 2', 'Region 3']
datos=[df_0, df_1, df_2]
predict_region={} #diccionario para almacenar 3 datafreme que retornó la función crear_modelo_entrenar_predecir
for datos, region in zip(datos,regiones):
  print(region)
  predict_region[region]=crear_modelo_entrenar_predecir(datos, state)
  print()




Region 1
RMSE 37.5794217150813
Volumen de barriles promedio predecidos 92.59256778438035
Volumen de barriles promedio reales 92.07859674082925

Region 2
RMSE 0.8897367737680648
Volumen de barriles promedio predecidos 68.76995145799754
Volumen de barriles promedio reales 68.77162424984647

Region 3
RMSE 39.958042459521614
Volumen de barriles promedio predecidos 95.087528122523
Volumen de barriles promedio reales 94.7489587172024



Observamos que la región 3 es la que tiene el mayor volumen de barriles extraidos, que es de 94.74, sin embargo  el menor error de prediccion lo tiene la region 2 con 0.88 miles de barriles.

In [24]:
#mostramos el diccionario que contiene 2 columnas datos reales y predicciones
predict_region

{'Region 1':        valor_real  prediccion_valores
 71751   10.038645           95.894952
 80493  114.551489           77.572583
 2655   132.603635           77.892640
 53233  169.072125           90.175134
 91141  122.325180           70.510088
 ...           ...                 ...
 12581  170.116726          103.037104
 18456   93.632175           85.403255
 73035  127.352259           61.509833
 63834   99.782700          118.180397
 43558  177.821022          118.169392
 
 [25000 rows x 2 columns],
 'Region 2':        valor_real  prediccion_valores
 70156   53.906522           54.238152
 42796   84.038886           83.217878
 36723    3.179103            2.971467
 79446   26.953261           25.916480
 93328  110.992147          111.341137
 ...           ...                 ...
 5864    26.953261           26.722422
 69826   80.859783           81.570895
 47036   84.038886           83.901769
 93329    0.000000           -1.259473
 93430   26.953261           26.169570
 
 [25000 r

## Cálculo de ganancias <a id='gains'></a>

Se calcula el punto de equilibrio, a partir de la cantidad mínima de barriles necesarios para recuperar la inversión, tanto a nivel global como por pozo. Esto proporciona un contexto financiero claro antes de evaluar las ganancias proyectadas.

In [25]:
#Variables establecidas
presupuesto=100000000   #cien millones de dolares presupuesto para 200 pozos, inversion inicial
incomes=4500     #Se genera 4,500 dolares por cada 1000 barriles extraídos
pozos=200  #nuevos que se desean explotar


In [26]:
# cuanto es el minimo de miles de barriles que necesito para reponer la inversion inicial 
volumen_minimo=presupuesto/incomes
print(volumen_minimo, 'miles de barriles en total para recuperar lo invertido')

22222.222222222223 miles de barriles en total para recuperar lo invertido


In [27]:
#calcula el volumen minimo que debe tener cada pozo para llegar al punto de equilibrio
volumen_minimo_pozo=volumen_minimo/pozos
print(volumen_minimo_pozo, 'miles de barriles debe generar cada pozo')

111.11111111111111 miles de barriles debe generar cada pozo


In [39]:
#Dentro de una región ubicamos a los 200 mejores pozos, para ello los ordenamos en forma descendente, hacemos la suma del volumen de
#miles de barriles generados y sacamos el margen de ganancia al miltiplicar el ingreso por todos los miles de barriles y restar la inversion
#selecciona los mejores pozos con base en la predicción, pero usa los valores reales para calcular las ganancias, lo cual es un excelente enfoque para evitar fuga de datos
def generacion(datos):
  mejores_lugares=datos.sort_values(by='prediccion_valores', ascending=False)['valor_real'].head(pozos)
  volumen_mejores_lugares=mejores_lugares.sum()
  margen=((volumen_mejores_lugares*incomes)-presupuesto)
  return margen

In [30]:
generacion(predict_region['Region 1'])

np.float64(33208260.43139851)

In [31]:
generacion(predict_region['Region 2'])

np.float64(24150866.966815114)

In [32]:
generacion(predict_region['Region 3'])

np.float64(25399159.45842947)

Se observa que el mayor margen de ganancia lo obtiene la región 1, teniendo en cuanta la capacidad de los mejores 200 pozos, a continuación se relaiza un ejercicio seleccionando 500 pozos al azar para obtener y se le realiza la tecnica de bootstraping con el objetivo de hacer un muestreo por repetición, se simulan 1,000 escenarios posibles generando muestras aleatorias a partir de un conjunto de datos original, calcular una métrica para cada una y guardar los resultados.

In [33]:
#Aplicamos herramienta de bootstraping para calcular la ganancia de un conjunto de pozos de petróleo seleccionados 
#Esta función toma al azar un conjunto de 500 ubicaciones dentro de una región y
#a ese conjunto se le calcula el beneficio que finalmente se agrega a la lista beneficio_muestra
# y ésto se repite 1000 veces

def modelo_bootstrap(datos, n_iteraciones=1000, state=state):
  generacion_muestra=[]
  for i in range(n_iteraciones):
    ubicaciones=datos.sample(n=500, replace=True)
    generacion_muestra.append(generacion(ubicaciones))
  generacion_muestra=pd.Series(generacion_muestra)
  return generacion_muestra

In [34]:
modelo_bootstrap(predict_region['Region 1'])

0      3.781275e+06
1      2.520976e+06
2      6.132341e+06
3      5.488377e+06
4      5.149196e+06
           ...     
995    2.613233e+06
996   -3.875887e+06
997    1.579504e+06
998    2.188819e+06
999    6.649111e+06
Length: 1000, dtype: float64

In [35]:
modelo_bootstrap(predict_region['Region 2'])

0      4.092665e+06
1      4.827245e+06
2      4.477060e+06
3      2.351691e+06
4      8.493925e+06
           ...     
995    8.116372e+06
996    3.071966e+06
997    4.527442e+06
998    6.082437e+06
999    6.753574e+06
Length: 1000, dtype: float64

In [36]:
modelo_bootstrap(predict_region['Region 3'])

0     -3.769165e+04
1      3.921738e+06
2      4.318804e+06
3      3.446845e+06
4      2.703410e+06
           ...     
995    2.157811e+06
996    5.244153e+06
997   -4.752451e+05
998    3.582789e+06
999    3.349284e+06
Length: 1000, dtype: float64

Se obtiene una lista con 1,000 ganancias posibles, por cada region, lo que permite evaluar el riesgo financiero del proyecto.

In [47]:
#Intervalo de confianza para concer la ganancia o perdida promedio
def metricas(generacion_muestra, region):
  nivel_confianza=0.95 #define umbral estadístico
  alpha=1-nivel_confianza
  generacion_promedio=generacion_muestra.mean() #Ganancia promedio de las 100 simulaciones
  riesgo_perdida=(generacion_muestra<0).mean()*100
  print("____________________")
  print("Las metricas de la Region", region)
  print("Ganancia promedio", generacion_promedio)
  print(f"Riesgo / Probabilidad de pérdida: {riesgo_perdida:.2f}% ")

In [48]:
for region, data in predict_region.items():
  margen_region=modelo_bootstrap(data)
  metricas(margen_region, region)

____________________
Las metricas de la Region Region 1
Ganancia promedio 3780227.2109624194
Riesgo / Probabilidad de pérdida: 7.80% 
____________________
Las metricas de la Region Region 2
Ganancia promedio 4583359.878413976
Riesgo / Probabilidad de pérdida: 1.20% 
____________________
Las metricas de la Region Region 3
Ganancia promedio 3380746.842051847
Riesgo / Probabilidad de pérdida: 10.10% 


# Conclusiones

Al evaluar los datos de los pozos en las tres distintas regiones, se pudo observar que la región 3 es la que tiene el mayor volumen de barriles extraidos, que es de 94.74, sin embargo  el menor error de prediccion lo tiene la region 2 con 0.88 miles de barriles. De igual forma al hacer un modelado de 1000 escenarios distintos con la muestrade 500 pozos, se observa que la region 2 es la que tiene un menor riesgo de perdida, al se solo del 1.20%.

Acerca del modelo utilizado un error tan bajo de 0.88 pocos errores catastróficos, el modelo mantiene una consistencia regular en sus predicciones, no comete errores gigantescos en ciertos registros que desestabilicen la métrica. Ademas de que el algoritmo de regresión lineal logró capturar bien la tendencia general de los datos sin sufrir de variaciones extremas.
